In [1]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers
import urllib3 

In [2]:
# Since the docker image uses elasticsearch 8.15.3, my local env also installs a es version 8, 
# as it is not compatible with version 9

# Connect to local Elasticsearch instance
client = Elasticsearch(
  "https://localhost:9200",
  basic_auth=("elastic", "mysecurepassword"),
  verify_certs=False
)

# Should provide a response with a cluster instance and name
client.info()

# Disable warnings caused by not using certificate verification
urllib3.disable_warnings()

c:\Users\mchrn\miniconda3\envs\recsys_exercise_2\Lib\site-packages\elasticsearch\_sync\client\__init__.py:326: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
c:\Users\mchrn\miniconda3\envs\recsys_exercise_2\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [3]:
client.indices.delete(index="books-index")

ObjectApiResponse({'acknowledged': True})

In [4]:
client.indices.create(
  index="books-index",
  settings={
    "number_of_shards": 4,
    "number_of_replicas": 2,
    "refresh_interval": -1,
    "analysis": {
      "analyzer": {
        "std_english": {"type": "standard", "stopwords": "_english_" }
      }
    }
  },
  mappings={
    "properties": {
      "title": {
       "type": "text",
      },
      "author": {
        "type": "text",
      },
      "date": {
        "type": "text" # Since date is not used in this exercise I set them as a string, cause I had trouble handling them otherwise with the format they are in
      },
      "summary": {
          "type": "text",
      }
    }
  }
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'books-index'})

In [5]:
# Create a helper function that reads the csv file and prepares it for a bulk insert into the index
def bulk_index(index_name="books-index"):
  df = pd.read_csv("../../data/books/booksummaries.csv")
  
  # Trying to load the dataset into Elasticsearch kept on throwing errors, looking at the data it seems that some entries are missing fields
  # so I use pandas to fill them in with an empty string value, since I handle all fields as strings.
  df_filled = df.fillna("")

  # Turn the csv file into a dictionary, so we can fetch the value for each row, by its column key
  # https://www.geeksforgeeks.org/python/pandas-dataframe-to_dict/
  df_dictionary = df_filled.to_dict(orient="records")
  
  # Use python generator function to load the dataset efficiently
  for record in df_dictionary:
    yield {
      "_index": index_name,
      "_id": record['wiki_id'],
      "_source": {
        "title": record['title'],
        "author": record['author'],
        "date": str(record['date']),
        "summary": record['summary']
      }
    }

# Code is inspired by: https://stackoverflow.com/questions/71889063/bulk-index-create-documents-with-elasticsearch-for-python 
# & https://www.geeksforgeeks.org/elasticsearch/using-the-elasticsearch-bulk-api-for-high-performance-indexing/
#helpers.bulk(client, bulk_index())
try:
  helpers.bulk(client, bulk_index())

except helpers.BulkIndexError as e:
  print(e)

In [6]:
# Update refresh_interval back to default
client.indices.put_settings(
  index="books-index",
  settings={
    "index": {
      "refresh_interval": "5s"
    }
  }
)

ObjectApiResponse({'acknowledged': True})

In [13]:
# Combined function that first queries the index and retrieves the top 10 results
# we extract documents ids for these and use the mtermvectors api to retrieve frequent terms
# this response is a lot of nested array, so we extract just the terms array and gather the
# keys from this array, which are the terms themselves.
# Returns a set of terms
def retrieve_terms_from_top_documents(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    size=10,
    query={
      "match": {
        "summary": query_phrase,
      }
    },
  )
  documents = resp['hits']['hits']

  # Create a list where I can store the id for each relevant document
  id_list = []

  # Go through each document gathered from the query and add its id to the list
  for document in documents:
    id_list.append(document["_id"])

  resp = client.mtermvectors(
    index=index_name, 
    body= {
      "ids": id_list,
      "parameters": {
        "fields": ["summary"],
        "term_statistics": True,
        "positions": False,
        "offsets": False,
        "filter": {
          "min_term_freq": 2,
          "min_word_length": 4         
        }
      }
    }
  )

  terms_array = resp['docs'][0]['term_vectors']['summary']['terms']

  # I extract each array key using the keys function, taken from: https://www.geeksforgeeks.org/python/python-get-dictionary-keys-as-a-list/
  terms = set(terms_array.keys())
  print("terms: ", terms)
  return terms

In [14]:
def original_query(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    query={
      "match": {
        "summary": query_phrase
      }
    }
  )
  print(resp)

# Same query expect we concenate each frequent term gathered, to expand the query 
def expanded_query(index_name, query_phrase, terms):
  new_query = query_phrase
  for term in terms:
    new_query = new_query + " " + term

  resp = client.search(
    index=index_name,
    query={
      "match": {
        "summary": new_query
      }
    }
  )
  print(resp)

In [15]:
original_query("books-index", "alien")
expanded_query("books-index", "alien", retrieve_terms_from_top_documents("books-index", "alien"))
print("-------------------------")
original_query("books-index", "ship")
expanded_query("books-index", "ship", retrieve_terms_from_top_documents("books-index", "ship"))
print("-------------------------")
original_query("books-index", "dream")
expanded_query("books-index", "dream", retrieve_terms_from_top_documents("books-index", "dream"))
print("-------------------------")
original_query("books-index", "zombie")
expanded_query("books-index", "zombie", retrieve_terms_from_top_documents("books-index", "zombie"))


{'took': 8, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 533, 'relation': 'eq'}, 'max_score': 6.7089205, 'hits': [{'_index': 'books-index', '_id': '7575811', '_score': 6.7089205, '_source': {'title': 'Earth Hive', 'author': 'Steve Perry', 'date': '1992', 'summary': ' The book begins with a routine space junk cleanup mission in Earth orbit, with a small derelict spaceship being prepped for de-orbit and burnup. The crew doing the cleanup investigate the ship, and, much to their horror, an alien xenomorph stowaway on the derelict manages to board the cleanup crew\'s ship, and kill them. The ship is destroyed when one of the crewmembers panics and collides the cleanup ship with the derelict. However, everything was captured by the cleanup ship\'s black box. A CIA-like organization, called the TIA, realizes the threat posed by the alien species, and is able to use the information in the black box to retrace the route of

## Comparison of the results of the original query and the expanded query
**alien**
- Original query
  - Total documents found: 533
  - Max_score: 6.7089205
- Expanded query
  - Total documents found: 1692
  - Max_score: 23.962824

**ship**
- Original query
  - Total documents found: 1351
  - Max_score: 5.0391626
- Expanded query
  - Total documents found: 1351
  - Max_score: 10.078325

**dream**
- Original query
  - Total documents found: 670
  - Max_score: 6.0306387
- Expanded query
  - Total documents found: 10000
  - Max_score: 133.30847

**zombie**
- Original query
  - Total documents found: 65
  - Max_score: 10.336001
- Expanded query
  - Total documents found: 6314
  - Max_score: 31.763193


Comparing the results of the queries, it can be seen that the maximum relevance score achieved by a document rises by a lot with the expanded query, and for most queries the total amount of documents that match the expanded query also increase. For most queries the most relevant document found in the original query, is not the same as with the expanded query. Should be noted that for ship the results are actually equal as the only frequent term found was ship, so the score is simply doubled because the final expanded query just ends up being "ship ship".


## Below is the step by step process I went through in trying to first gather term frequencies and then gather the terms from these, and finally making expanded queries.
I was unsure if the process was supposed to be included in the code, if it is not of interest just ignore what is included below here

In [16]:
# Pseudo-relevance feedback = I need to take informative terms from top k documents and add them to a query that is then submitted to the index
def retrieve_relevant_docs(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    size=10,
    query={
      "match": {
        "summary": query_phrase,
      }
    },
  )
  print(resp)

retrieve_relevant_docs("books-index", "alien")

{'took': 7, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 533, 'relation': 'eq'}, 'max_score': 6.7089205, 'hits': [{'_index': 'books-index', '_id': '7575811', '_score': 6.7089205, '_source': {'title': 'Earth Hive', 'author': 'Steve Perry', 'date': '1992', 'summary': ' The book begins with a routine space junk cleanup mission in Earth orbit, with a small derelict spaceship being prepped for de-orbit and burnup. The crew doing the cleanup investigate the ship, and, much to their horror, an alien xenomorph stowaway on the derelict manages to board the cleanup crew\'s ship, and kill them. The ship is destroyed when one of the crewmembers panics and collides the cleanup ship with the derelict. However, everything was captured by the cleanup ship\'s black box. A CIA-like organization, called the TIA, realizes the threat posed by the alien species, and is able to use the information in the black box to retrace the route of

In [17]:
# Pseudo-relevance feedback = I need to take informative terms from top k documents and add them to a query that is then submitted to the index
def retrieve_relevant_docs(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    size=10,
    query={
      "match": {
        "summary": query_phrase,
      }
    },
  )
  # From the response above we I can see that the ids are stored in an array called hits that is inside an array called hits, so I first extract just the array containing the ids
  return resp['hits']['hits']

documents = retrieve_relevant_docs("books-index", "alien")
print(documents)

# Create a list where I can store the id for each relevant document
id_list = []
# Go through each document gathered from the query and add its id to the list
for document in documents:
  id_list.append(document["_id"])

# Verify that we have a list containing ids
print (id_list)

[{'_index': 'books-index', '_id': '7575811', '_score': 6.7089205, '_source': {'title': 'Earth Hive', 'author': 'Steve Perry', 'date': '1992', 'summary': ' The book begins with a routine space junk cleanup mission in Earth orbit, with a small derelict spaceship being prepped for de-orbit and burnup. The crew doing the cleanup investigate the ship, and, much to their horror, an alien xenomorph stowaway on the derelict manages to board the cleanup crew\'s ship, and kill them. The ship is destroyed when one of the crewmembers panics and collides the cleanup ship with the derelict. However, everything was captured by the cleanup ship\'s black box. A CIA-like organization, called the TIA, realizes the threat posed by the alien species, and is able to use the information in the black box to retrace the route of the derelict ship. A decision is made to send a Colonial Marine expedition to the origin planet of the derelict ship, presumably the home planet of the alien species. Wilks, a battle-h

In [18]:
# Use the multiple termvectors api endpoint to get common terms from the summary field. Terms have to be off length 4 and appear atleast 3 times (these could be modified)
# Word length is mainly to filter out some fillers like the, and, etc. Term frequency just so we dont simply get every word that appears once, should be adjusted up or down
# based on the amount of terms we receive back
def get_terms(index_name, id_list = id_list):
  resp = client.mtermvectors(
    index=index_name, 
    body= {
      "ids": id_list,
      "parameters": {
        "fields": ["summary"],
        "term_statistics": True,
        "positions": False,
        "offsets": False,
        "filter": {
          "min_term_freq": 2,
          "min_word_length": 4         
        }
      }
    }
  )
  return resp

docs = get_terms("books-index", id_list)
print(docs)

# In the print statement below I can see that "terms" is an array containing each terms as a key that maps to information on the term like its frequency -
# this array is located in several other arrays, so to extract just the terms array I just trial and errored it, as I could not find a good way to do it in python
# though I am quite sure there is a better way
terms_array = docs['docs'][0]['term_vectors']['summary']['terms']

# Print to verify I now have an array, where each key is a term
print(terms_array)

# I extract each array key using the keys function, taken from: https://www.geeksforgeeks.org/python/python-get-dictionary-keys-as-a-list/
terms = set(terms_array.keys())

# Finally I print my set of terms, to verify that it does only contain words.
print(terms)


{'docs': [{'_index': 'books-index', '_id': '6063708', '_version': 1, 'found': True, 'took': 1, 'term_vectors': {'summary': {'field_statistics': {'sum_doc_freq': 858573, 'doc_count': 4149, 'sum_ttf': 1830930}, 'terms': {'alien': {'doc_freq': 127, 'ttf': 224, 'term_freq': 2, 'score': 8.957666}, 'military': {'doc_freq': 260, 'ttf': 430, 'term_freq': 2, 'score': 7.532686}, 'weapon': {'doc_freq': 119, 'ttf': 190, 'term_freq': 2, 'score': 9.086743}}}}}, {'_index': 'books-index', '_id': '6027087', '_version': 1, 'found': True, 'took': 1, 'term_vectors': {'summary': {'field_statistics': {'sum_doc_freq': 816252, 'doc_count': 4073, 'sum_ttf': 1687855}, 'terms': {'alien': {'doc_freq': 123, 'ttf': 204, 'term_freq': 2, 'score': 8.984198}, 'doctor': {'doc_freq': 251, 'ttf': 577, 'term_freq': 2, 'score': 7.565903}, 'peri': {'doc_freq': 4, 'ttf': 6, 'term_freq': 3, 'score': 23.10883}}}}}, {'_index': 'books-index', '_id': '14216872', '_version': 1, 'found': True, 'took': 1, 'term_vectors': {'summary': 

In [19]:
def non_expanded_query(index_name, query_phrase):
  resp = client.search(
    index=index_name,
    query={
      "match": {
        "summary": query_phrase
      }
    }
  )
  print(resp)

def expanded_query(index_name, query_phrase, terms):
  new_query = query_phrase
  for term in terms:
    new_query = new_query + " " + term

  resp = client.search(
    index=index_name,
    query={
      "match": {
        "summary": new_query
      }
    }
  )
  print(resp)

non_expanded_query("books-index", "alien")
expanded_query("books-index", "alien", terms)

{'took': 5, 'timed_out': False, '_shards': {'total': 4, 'successful': 4, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 533, 'relation': 'eq'}, 'max_score': 6.7089205, 'hits': [{'_index': 'books-index', '_id': '7575811', '_score': 6.7089205, '_source': {'title': 'Earth Hive', 'author': 'Steve Perry', 'date': '1992', 'summary': ' The book begins with a routine space junk cleanup mission in Earth orbit, with a small derelict spaceship being prepped for de-orbit and burnup. The crew doing the cleanup investigate the ship, and, much to their horror, an alien xenomorph stowaway on the derelict manages to board the cleanup crew\'s ship, and kill them. The ship is destroyed when one of the crewmembers panics and collides the cleanup ship with the derelict. However, everything was captured by the cleanup ship\'s black box. A CIA-like organization, called the TIA, realizes the threat posed by the alien species, and is able to use the information in the black box to retrace the route of